In [1]:
from chromadb.config import Settings
import chromadb

from utils.var import content

client = chromadb.PersistentClient(path= content["vectorDBPath"])

In [2]:
collection = client.get_collection(
    name="job_roles_DB"
)
collection

Collection(name=job_roles_DB)

In [3]:
res = collection.get(
    where={"roleID" : "computer_vision_engineer"},
    include=["embeddings" , "documents"]
)
res

{'ids': ['computer_vision_engineer_chunk_0',
  'computer_vision_engineer_chunk_1',
  'computer_vision_engineer_chunk_2',
  'computer_vision_engineer_chunk_3',
  'computer_vision_engineer_chunk_4',
  'computer_vision_engineer_chunk_5'],
 'embeddings': array([[-0.02222971,  0.01182222,  0.02249167, ..., -0.01899873,
          0.04034064, -0.01841361],
        [-0.1109584 , -0.1211109 ,  0.00794348, ...,  0.05095052,
         -0.0315654 , -0.02031076],
        [-0.06658333, -0.14422604, -0.02299616, ...,  0.01704387,
          0.03196792,  0.0299962 ],
        [-0.0622731 , -0.10466916, -0.00063204, ...,  0.0473175 ,
          0.04626534,  0.01956989],
        [-0.0349458 , -0.02718824, -0.02522505, ..., -0.01317053,
         -0.07731739,  0.01395973],
        [-0.01609409,  0.02921227, -0.01476897, ...,  0.00942456,
         -0.07187848,  0.0146528 ]], shape=(6, 384)),
 'documents': ['title:Computer Vision Engineer\n\n            description:Specializes in image and video processing',
  

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from utils.var import content
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_CV_embeddings(CVPath):
    loader = PyPDFLoader(CVPath)
    CV = loader.load()

    text = ""
    for i in range(len(CV)):
      text += CV[i].page_content
    
    textSplitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50,
                separators=["\n\n", "\n", ".", " ", ""]
            )
    textSplitted = textSplitter.split_text(text)

    embeddingModel = SentenceTransformer("all-MiniLM-L6-v2")
    vectors = embeddingModel.encode(
                textSplitted,
                batch_size=16,
                normalize_embeddings=True,
                show_progress_bar=True
            ).tolist()
    
    return vectors

    

   


c:\Users\01\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def text_extractor(filePath):
    loader = PyPDFLoader(filePath)
    CV = loader.load()

    text = ""
    for i in range(len(CV)):
      text += CV[i].page_content

    return text

In [5]:
Vectors = get_CV_embeddings(content["exCV"])
Vectors

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 199.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.06it/s]


[[-0.04541822522878647,
  0.02113993465900421,
  0.03065764158964157,
  -0.009608421474695206,
  -0.04298512265086174,
  -0.09879095107316971,
  0.02185169979929924,
  0.02124086208641529,
  -0.021548645570874214,
  -0.033404458314180374,
  -0.10992176085710526,
  0.028109880164265633,
  0.09708271920681,
  -0.060816433280706406,
  -0.009393549524247646,
  0.062366124242544174,
  0.002005739603191614,
  -0.05457833781838417,
  -0.01269024983048439,
  -0.13662195205688477,
  -0.02330387569963932,
  0.012653561308979988,
  -0.06909883767366409,
  0.003747115610167384,
  -0.023807309567928314,
  0.04086246341466904,
  0.0641975998878479,
  -0.053599413484334946,
  0.007015395909547806,
  -0.01471547782421112,
  0.01929238624870777,
  0.01395623292773962,
  0.12755022943019867,
  -0.021154316142201424,
  -0.07015939801931381,
  0.03419065102934837,
  -0.02147844061255455,
  -0.01845228485763073,
  0.0960685983300209,
  0.04561866447329521,
  -0.013659712858498096,
  0.0034648270811885595,


In [10]:
text = text_extractor(content["exCV"])

In [19]:
from openai import OpenAI

OpenAIClient = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=content["HFTOKEN"],
)

def getRole(CVText):

    try:

        prompt_roles = f'''You are an ATS classifier.

Your task:
- Identify the closest matching job role from the allowed list.
- If no match exists, return "false".

Allowed roles:
computer_vision_engineer
nlp_engineer
devops_engineer
web_developer
fullstack_engineer
frontend_engineer
backend_engineer
data_analyst
data_scientist
ai_engineer
ml_engineer

Return ONLY one role name or "false"..
        '''

        messages=[
            {"role": "system", "content": prompt_roles},
            {"role": "user", "content": CVText}
        ]
        completion = OpenAIClient.chat.completions.create(
        model="openai/gpt-oss-20b:groq",
        messages = messages,
        temperature=0
            )

        role = completion.choices[0].message.content.strip()
        return role
    
    except Exception as e:
        print(e)

In [13]:
role = getRole(text)
role

'ai_engineer'

In [14]:
type(role)

str

In [8]:
type(Vectors)

list

In [16]:
def loadRoleVectors(collectionName:str , directoryPath:str , role:str):
    try:

        client = chromadb.PersistentClient(path= directoryPath)
        collection = client.get_collection(
        name=collectionName
        )

        roleVectors  = collection.get(
        where={"roleID" : role},
        include=["embeddings" , "documents"]
        )

        return roleVectors

    except Exception as e:
        print(e)

roleVectors = loadRoleVectors(collectionName="job_roles_DB" , directoryPath=content["vectorDBPath"] , role=role)
roleVectors
    
    

{'ids': ['ai_engineer_chunk_0',
  'ai_engineer_chunk_1',
  'ai_engineer_chunk_2',
  'ai_engineer_chunk_3',
  'ai_engineer_chunk_4',
  'ai_engineer_chunk_5'],
 'embeddings': array([[-6.23144843e-02,  5.43512078e-03,  3.44899185e-02, ...,
          6.16737939e-02,  3.70626189e-02, -1.18690259e-04],
        [-4.90983948e-02, -1.46089941e-01,  2.77317297e-02, ...,
          6.38224855e-02, -2.54084077e-02,  2.53843646e-02],
        [-3.81977819e-02, -8.41370225e-02, -2.35537719e-02, ...,
          3.28685157e-02,  6.18162900e-02,  7.91408196e-02],
        [-1.14040030e-02, -6.19952865e-02,  1.83041934e-02, ...,
          9.32496786e-03, -4.96738665e-02,  1.43124992e-02],
        [-9.72115248e-03, -2.24603098e-02,  4.04374711e-02, ...,
          6.87921196e-02, -1.55124292e-02, -3.61275189e-02],
        [ 5.92833105e-03, -7.98124224e-02,  7.14597665e-03, ...,
          7.18260333e-02, -1.06092189e-02,  6.10096641e-02]],
       shape=(6, 384)),
 'documents': ['title:AI Engineer\n\n          

In [ ]:
def similaritySearch_LLM(CVText , RoleText , role):
    try:
        SystemPrompt = '''you are automating CVs for comapny. Your job is to Rank provided CV text with requirements of that job. 
        return only score in range of 1 to 100, '''
        UserPrompt = f'''
        text of CV:{CVText} ,,,, 
        text of requriements for {role}:{RoleText}'''

        messages = [
            {"role":"system" , "content":SystemPrompt},
            {"role":"user" , "content":UserPrompt}
        ]

        completion = OpenAIClient.chat.completions.create(
        model="openai/gpt-oss-20b:groq",
        messages = messages,
        temperature=0
            )
        
        score = completion.choices[0].message.content.strip()
        return score
        
        
    except Exception as e:
        print(e)

In [21]:
score = similaritySearch_LLM(text , roleVectors['documents'] , role)

In [22]:
score

'55'